In [ ]:
# verify_checkpoints.ipynb -- batch integrity check after the duelling-writer
# incident (two VMs training one combo shared a fixed .tmp; interleaved writes
# could publish CORRUPT .pt files; fixed in d636f8a, but files written before
# the fix need auditing).
#
# CHECK  : every combo's latest/best (+ probe_ckpts/*.pt) as zip archives with
#          CRC verification (torch.save files are zips; interleaved writes
#          break structure/CRC) -- falls back to torch.load for legacy files.
#          manifest.json must parse; stray *.tmp* staging leftovers are listed.
# REPAIR : APPLY=True deletes corrupt files. Fallbacks then engage naturally:
#          corrupt best -> battery/eval use latest; corrupt latest -> resume
#          from best is impossible so if NO valid full checkpoint remains on a
#          DONE combo, its done.json + manifest are removed too -> the combo
#          becomes claimable again and simply retrains. Corrupt probe
#          snapshots are deleted (their queue markers will fail loudly at
#          drain -- re-emit is impossible, that history point is lost).
import json, os, zipfile
from pathlib import Path

APPLY = False        # dry-run first; review, then True + re-run
CHECK_PROBE_CKPTS = True
CLEAN_STRAY_TMP = True
REPO = "/workspace/stable-query-latent"
OUT_DIR = "VICReg_review/heads/cloud_full_sweep_a100"
root = Path(REPO) / OUT_DIR

FULL = ('vicreg_review_h5_latest.pt', 'vicreg_review_h5_best.pt')
MANIFEST = 'vicreg_review_h5_manifest.json'


def pt_ok(p):
    """True if the torch file is structurally sound (zip CRC pass)."""
    try:
        with zipfile.ZipFile(p) as z:
            return z.testzip() is None
    except zipfile.BadZipFile:
        try:                                  # legacy (pre-zip) torch format
            import torch
            torch.load(p, map_location='cpu', weights_only=False)
            return True
        except Exception:
            return False
    except Exception:
        return False


def is_done(d):
    if (d / 'done.json').exists():
        return True
    try:
        return json.loads((d / MANIFEST).read_text(encoding='utf-8')).get('status') == 'done'
    except Exception:
        return False


corrupt = []          # (path, kind)
bad_manifest = []
stray_tmp = []
checked = 0

for d in sorted(root.iterdir()):
    if not d.is_dir() or d.name in ('VM_parallel', 'probe_queue', '_coord_probe',
                                    'final_best_eval', 'raw_test_data'):
        continue
    for name in FULL:
        f = d / name
        if f.exists():
            checked += 1
            if not pt_ok(f):
                corrupt.append((f, 'full'))
    if CHECK_PROBE_CKPTS and (d / 'probe_ckpts').is_dir():
        for f in sorted((d / 'probe_ckpts').glob('*.pt')):
            checked += 1
            if not pt_ok(f):
                corrupt.append((f, 'probe'))
    mf = d / MANIFEST
    if mf.exists():
        try:
            json.loads(mf.read_text(encoding='utf-8'))
        except Exception:
            bad_manifest.append(mf)
    for f in d.glob('*.tmp*'):
        stray_tmp.append(f)
    if (d / 'probe_ckpts').is_dir():
        for f in (d / 'probe_ckpts').glob('*.tmp*'):
            stray_tmp.append(f)

print(f'checked {checked} .pt file(s) under {root}')
print(f'CORRUPT .pt      : {len(corrupt)}')
for f, kind in corrupt:
    print(f'  [{kind:5}] {f.relative_to(root)}')
print(f'corrupt manifest : {len(bad_manifest)}')
for f in bad_manifest:
    print(f'  {f.relative_to(root)}')
print(f'stray tmp files  : {len(stray_tmp)}'
      + (f'  ({sum(f.stat().st_size for f in stray_tmp) / 2**20:.0f} MiB)' if stray_tmp else ''))
for f in stray_tmp[:10]:
    print(f'  {f.relative_to(root)}')
if len(stray_tmp) > 10:
    print(f'  ...... ({len(stray_tmp) - 10} more)')

if not APPLY:
    print()
    print('DRY-RUN: nothing deleted. Set APPLY=True and re-run to repair.')
else:
    deleted = reset = 0
    touched_dirs = set()
    for f, kind in corrupt:
        try:
            f.unlink()
            deleted += 1
            if kind == 'full':
                touched_dirs.add(f.parent)
        except OSError:
            pass
    for d in sorted(touched_dirs):
        has_valid_full = any((d / n).exists() and pt_ok(d / n) for n in FULL)
        if is_done(d) and not has_valid_full:
            for name in ('done.json', MANIFEST):
                try:
                    (d / name).unlink()
                except OSError:
                    pass
            reset += 1
            print(f'RESET for retrain (done but no valid checkpoint left): {d.name}')
    tmp_n = 0
    if CLEAN_STRAY_TMP:
        for f in stray_tmp:
            try:
                f.unlink()
                tmp_n += 1
            except OSError:
                pass
    print()
    print(f'summary: deleted {deleted} corrupt file(s); reset {reset} combo(s) '
          f'for retrain; removed {tmp_n} stray tmp file(s).')
    if reset:
        print('reset combos are claimable again -- any running/next training '
              'session picks them up automatically (FULL-N first ordering applies).')
